# **Solemne 1 — Minería de Datos 2026**

# 1.- Obtencion de datos
## ¿qué espero encontrar?
En la tabla de 'health_nutrition_population' espero encontrar referencias a la salud nutricional de la poblacion en algun pais/ciudad especificado en la tabla.

In [22]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="yo-si-489418")

dataset_ref = client.dataset("world_bank_health_population", project="bigquery-public-data") #Dentro del la bigquery publica, busca el proyecto del banco mundial

tables = list(client.list_tables(dataset_ref))

table_ref = dataset_ref.table("health_nutrition_population")

query = """
SELECT *
FROM `bigquery-public-data.world_bank_health_population.health_nutrition_population`
ORDER BY year DESC
"""


dry_run_config = bigquery.QueryJobConfig(dry_run=True)

dry_run_query_job = client.query(query, job_config=dry_run_config) #Estas lineas de codigo me mostraran cuando pesaran los datos que estare consultando.

print()
print("This query will process {} bytes.".format(dry_run_query_job.total_bytes_processed))



safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10) #Esto limita la cantidad de bytes que podria procesar (quizas no necesario pero no esta de mas)
query_job = client.query(query, job_config=safe_config)


Pandass = query_job.to_dataframe()       #transforma la query a un dataframe de pandas

Pandass

c:\Users\joshe\Documents\U weas\GIT\venv\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)



This query will process 310099813 bytes.


,country_name,country_code,indicator_name,indicator_code,value,year
0,Belarus,BLR,"Population ages 70-74, male (% of male populat...",SP.POP.7074.MA.5Y,2.252892,2020
1,Other small states,OSS,"Population ages 05-09, female (% of female pop...",SP.POP.0509.FE.5Y,10.482102,2020
2,Vietnam,VNM,"Population ages 35-39, male",SP.POP.3539.MA,1.000000,2020
3,Indonesia,IDN,People with basic handwashing facilities inclu...,SH.STA.HYGN.ZS,94.107068,2020
4,New Zealand,NZL,"Population ages 05-09, male",SP.POP.0509.MA,1.000000,2020
...,...,...,...,...,...,...
3174991,Lesotho,LSO,"Age population, age 18, male, interpolated",SP.POP.AG18.MA.IN,1.000000,1960
3174992,Malaysia,MYS,"Age population, age 02, female, interpolated",SP.POP.AG02.FE.IN,1.000000,1960
3174993,Gibraltar,GIB,Rural population (% of total population),SP.RUR.TOTL.ZS,1.000000,1960
3174994,Mozambique,MOZ,"Age population, age 04, female, interpolated",SP.POP.AG04.FE.IN,1.000000,1960


# Analisis de Datos

---

Primero dare los datos pedidos como outpout de codigo sin filtrar y luego el filtrado de datos para trabajar de forma eficiente

In [76]:
print(f"Filas: {Pandass.shape[0]:,} | Columnas: {Pandass.shape[1]}")
print("Tipos:", ", ".join([f"{col}: {dtype}" for col, dtype in Pandass.dtypes.items()]))
print(f"Rango temporal: {Pandass['year'].min()} - {Pandass['year'].max()}")

nulos = Pandass.isnull().sum()
print("Nulos:", ", ".join([f"{col}: {val:,}" for col, val in nulos.items()]))


Filas: 84,879 | Columnas: 4
Tipos: country_name: object, indicator_name: object, value: float64, year: Int64
Rango temporal: 2000 - 2020
Nulos: country_name: 0, indicator_name: 0, value: 0, year: 0


---
Ahora hare un diccionario con los paises de latinoamerica para filtrar despues
, ya que este es un lugar mas cercano y mas de interes que paises de otro continente o que hablen otro idioma (por el entendimiento mas que nada)

In [39]:
latam = [
    'Argentina', 'Bolivia', 'Brasil', 'Chile', 'Colombia',
    'Costa Rica', 'Cuba', 'Ecuador', 'El Salvador', 'Guatemala',
    'Honduras', 'México', 'Nicaragua', 'Panamá', 'Paraguay',
    'Perú', 'República Dominicana', 'Uruguay', 'Venezuela'
]

Luego usare varios filtros y descartaremos algunas columnas, por ejemplo descartaremos 'country_code' y 'indicator_code' ya que esto no nos entrega datos relevantes para el analisis.

Tambien filtraremos los años, donde se veran los datos tomas entre los años 2000 y 2020, esto porque pienso que los datos atras del 2000 pueden verse afectados debido a la falta de tecnologia y concexion que pudo haber en años posteriores

In [71]:
Pandass_act = Pandass[['country_name', 'indicator_name', 'value', 'year']]
Pandass_act = Pandass_act[(Pandass_act['year'].between(2000, 2020) & Pandass_act['country_name'].isin(latam))]


print(Pandass_act['country_name'].unique())
print()
print(f"Filas: {Pandass_act.shape[0]:,} | Columnas: {Pandass_act.shape[1]}")
print("Tipos:", ", ".join([f"{col}: {dtype}" for col, dtype in Pandass_act.dtypes.items()]))
print(f"Rango temporal: {Pandass_act['year'].min()} - {Pandass_act['year'].max()}")

nulos = Pandass_act.isnull().sum()
print("Nulos:", ", ".join([f"{col}: {val:,}" for col, val in nulos.items()]))


['El Salvador' 'Ecuador' 'Guatemala' 'Nicaragua' 'Colombia' 'Argentina'
 'Bolivia' 'Cuba' 'Uruguay' 'Chile' 'Costa Rica' 'Honduras' 'Paraguay']

Filas: 84,879 | Columnas: 4
Tipos: country_name: object, indicator_name: object, value: float64, year: Int64
Rango temporal: 2000 - 2020
Nulos: country_name: 0, indicator_name: 0, value: 0, year: 0


Viendo estos datos los cuales fueron filtrado a datos mas actuales (Del 2000 en adelante) esto para tener una mayor fiabilidad en la compilacion de estos (tambien porque no existen nulos), espero que tengan una calidad decente para poder analisarlos de forma optima. Con un alcance directamente sobre la poblacion latinoamerica, donde busco la tasa de mortalidad en suicidios en hombres y mujeres., como la cantidad de filas no es tan grande podria ser que los datos a analizar no nos den una estadistica muy correcta.

---

Entonces ahora hare un filtro donde el las columnas 'indicator_name' tengan la palabra 'suicide'


In [82]:
suicidios = Pandass_act[Pandass_act['indicator_name'].str.contains('suicide', case=False, na=False)]

#Uso todos estos datos para separar el termino suicidio dentro de la columna "indicator_name"

suicidios = suicidios.copy()            #me arrojaba un warning asi que con esto pude solucionarlo...

def normalizar(x):
    return (x - x.min()) / (x.max() - x.min())                  #funcion para normalizar los valores de la tabla

suicidios['valores_norma'] = suicidios.groupby('country_name')['value'].transform(normalizar)

suicidios = suicidios.sort_values(['country_name', 'year'])
suicidios

,country_name,indicator_name,value,year,valores_norma
1382616,Argentina,"Suicide mortality rate, male (per 100,000 male...",15.2,2000,0.893082
1386129,Argentina,"Suicide mortality rate, female (per 100,000 fe...",3.4,2000,0.150943
1401605,Argentina,"Suicide mortality rate (per 100,000 population)",9.1,2000,0.509434
1292967,Argentina,"Suicide mortality rate, male (per 100,000 male...",16.9,2001,1.000000
1297674,Argentina,"Suicide mortality rate (per 100,000 population)",10.3,2001,0.584906
...,...,...,...,...,...
151931,Uruguay,"Suicide mortality rate (per 100,000 population)",21.2,2018,0.599407
165786,Uruguay,"Suicide mortality rate, male (per 100,000 male...",34.7,2018,1.000000
58736,Uruguay,"Suicide mortality rate (per 100,000 population)",21.2,2019,0.599407
70079,Uruguay,"Suicide mortality rate, male (per 100,000 male...",34.5,2019,0.994065


In [91]:
dic_paises = {}
for pais in suicidios["country_name"].unique():
    dic_paises[pais] = suicidios[suicidios["country_name"] == pais]
    print(f"{pais}: {len(dic_paises[pais])} filas")


tabla_hori = pd.DataFrame({pais: dic_paises[pais]['valores_norma'].values for pais in dic_paises})

tabla_hori

Argentina: 60 filas
Bolivia: 60 filas
Chile: 60 filas
Colombia: 60 filas
Costa Rica: 60 filas
Cuba: 60 filas
Ecuador: 60 filas
El Salvador: 60 filas
Guatemala: 60 filas
Honduras: 60 filas
Nicaragua: 60 filas
Paraguay: 60 filas
Uruguay: 60 filas


,Argentina,Bolivia,Chile,Colombia,Costa Rica,Cuba,Ecuador,El Salvador,Guatemala,Honduras,Nicaragua,Paraguay,Uruguay
0,0.893082,0.000000,0.862245,0.984848,0.786260,0.686441,0.375000,0.363636,0.145349,0.406250,0.902778,0.168831,0.774481
1,0.150943,0.756410,0.096939,0.242424,0.412214,0.372881,0.520833,0.566434,0.941860,0.203125,0.361111,0.376623,0.000000
2,0.509434,0.474359,0.474490,0.000000,0.038168,1.000000,0.229167,0.167832,0.534884,0.000000,0.625000,0.272727,0.136499
3,1.000000,0.987179,0.862245,0.984848,0.320611,0.309322,0.569444,0.405594,0.000000,0.203125,0.319444,0.272727,0.400593
4,0.584906,0.448718,0.096939,0.227273,0.603053,0.605932,0.250000,0.188811,0.162791,0.406250,0.583333,0.194805,0.157270
5,0.000000,0.717949,0.474490,0.590909,0.038168,0.902542,0.409722,0.000000,0.563953,0.000000,0.847222,0.337662,0.658754
6,0.981132,0.461538,0.438776,0.560606,0.832061,0.313559,0.284722,0.153846,0.552326,0.406250,0.263889,0.285714,0.000000
7,0.201258,0.717949,0.000000,0.212121,0.000000,0.851695,0.673611,0.300699,0.168605,0.000000,0.000000,0.376623,0.557864
8,0.578616,0.961538,0.790816,0.924242,0.000000,0.580508,0.479167,0.461538,0.953488,0.203125,0.875000,0.337662,0.219585
9,0.937107,0.448718,0.443878,0.545455,0.526718,0.542373,0.743056,0.160839,0.168605,0.218750,0.638889,0.311688,0.181009
